Try the GX241 GSIOC response code

NOTE: Running this code means the VB script from gilson no longer works! Check that the red light goes on the pump

In [1]:
import ftd2xx
print(ftd2xx.listDevices())

[b'G0SX4P41']


In [2]:
import ftd2xx
import time

class Gilson4020PumpUSB:
    def __init__(self, ftdi_serial=None):
        self.device = self._connect_to_device(ftdi_serial)
        self._configure_device()

    def _connect_to_device(self, ftdi_serial):
        devices = ftd2xx.listDevices()
        if not devices:
            raise RuntimeError("No FTDI devices found.")

        decoded_devices = [d.decode() if isinstance(d, bytes) else d for d in devices]
        if ftdi_serial is None:
            ftdi_serial = decoded_devices[0]
        elif ftdi_serial not in decoded_devices:
            raise RuntimeError(f"FTDI device with serial '{ftdi_serial}' not found. Available: {decoded_devices}")

        try:
            return ftd2xx.openEx(ftdi_serial.encode('ascii'), 1)
        except ftd2xx.DeviceError as e:
            raise RuntimeError(f"Failed to open FTDI device '{ftdi_serial}': {e}")

    def _configure_device(self):
        self.device.setTimeouts(1000, 1000)
        self.device.purge()
        self.device.setDtr()
        self.device.setRts()
        self.device.setLatencyTimer(16)
        self.device.setUSBParameters(4096, 4096)
        self.device.setChars(0, False, 0, False)
        self.device.setFlowControl(0x0100, 0x11, 0x13)
        self.device.setBitMode(0, 0x00)
        self.device.setBitMode(0, 0x40)

    def send_command(self, cmd, read_timeout=1.0):
        full_cmd = (cmd + "\r").encode('ascii')
        print(f"[DEBUG] Sending command: {repr(full_cmd)}")
        self.device.write(full_cmd)
        self.device.purge()  # Clear input buffer before reading
        time.sleep(0.2)  # Allow time for response

        start_time = time.time()
        response = b''
        while time.time() - start_time < read_timeout:
            chunk = self.device.read(1024)
            if chunk:
                print(f"[DEBUG] Received chunk: {repr(chunk)}")
                response += chunk
                if b'\r' in chunk or b'\n' in chunk:
                    break
            else:
                time.sleep(0.05)

        if not response:
            print("[DEBUG] No response received from device.")
            raise RuntimeError("No response received from device.")

        decoded = response.decode('ascii', errors='ignore').strip()
        print(f"[DEBUG] Final response: {repr(decoded)}")
        return decoded

    def identify(self):
        return self.send_command('Identify')

    def home(self):
        return self.send_command('p')

    def get_serial_number(self):
        self.send_command('.sn')
        time.sleep(0.2)
        return self.send_command('.')

    def reset(self):
        return self.send_command('$')

    def close(self):
        self.device.close()


In [3]:
pump = Gilson4020PumpUSB()

In [4]:
print("Identify:", pump.identify())
#print("Serial:", pump.get_serial_number())
#print("Home:", pump.home())
pump.close()

[DEBUG] Sending command: b'Identify\r'
[DEBUG] No response received from device.


RuntimeError: No response received from device.

In [ ]:
pump.close()

In [ ]:
pump.device.write(b'\xFF')
time.sleep(0.5)
resp = pump.device.read(1024)
print("[DEBUG] Wake-up response:", repr(resp))


In [ ]:
pump.device.write(b'%')
time.sleep(0.5)
resp = pump.device.read(1024)
print("[DEBUG] Raw response:", repr(resp))


In [ ]:
pump.device.write(b'Aspirate Valve=N Volume=100.000 FlowRate=1.000\r')
time.sleep(0.5)
resp = pump.device.read(1024)
print("[DEBUG] Full-text command response:", repr(resp))
